# Raw Code from hugging face

In [ ]:
# Requires transformers>=4.51.0
import torch
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoTokenizer, AutoModel


def last_token_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'

# Each query must come with a one-sentence instruction that describes the task
task = 'Given a web search query, retrieve relevant passages that answer the query'

queries = [
    get_detailed_instruct(task, 'What is the capital of China?'),
    get_detailed_instruct(task, 'Explain gravity')
]
# No need to add instruction for retrieval documents
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun."
]
input_texts = queries + documents

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-4B', padding_side='left')
model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-4B')

# We recommend enabling flash_attention_2 for better acceleration and memory saving.
#model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-4B', attn_implementation="flash_attention_2", torch_dtype=torch.float16).cuda()

max_length = 8192

# Tokenize the input texts
batch_dict = tokenizer(
    input_texts,
    padding=True,
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
)
batch_dict.to(model.device)
outputs = model(**batch_dict)
embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])

# normalize embeddings
embeddings = F.normalize(embeddings, p=2, dim=1)
scores = (embeddings[:2] @ embeddings[2:].T)
print(scores.tolist())
# [[0.7534257769584656, 0.1146894246339798], [0.03198453038930893, 0.6258305311203003]]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[[0.75390625, 0.11376953125], [0.03125, 0.62109375]]


# Modified Code

In [4]:
trial_list = [
    # Marvel Cinematic Universe
    "Iron Man",
    "The Avengers",
    "Spider-Man: No Way Home",
    "Avengers: Endgame",
    
    # Tom Cruise
    "Top Gun: Maverick",
    "Mission: Impossible - Fallout",
    "Jerry Maguire",
    "Minority Report",
    
    # James Bond (007)
    "Casino Royale",
    "Skyfall",
    "GoldenEye",
    "Goldfinger",
    
    # SpongeBob SquarePants
    "The SpongeBob SquarePants Movie",
    "The SpongeBob Movie: Sponge Out of Water",
    "The SpongeBob Movie: Sponge on the Run"
]


In [ ]:
import torch 
import torch
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoTokenizer, AutoModel


def last_token_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]
    

max_length = 8192

# Tokenize the input texts
batch_dict = tokenizer(
    trial_list,
    padding=True,
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
)

batch_dict.to(model.device)
outputs = model(**batch_dict)
embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
embeddings = F.normalize(embeddings, p=2, dim=1)

In [13]:
import numpy as np

list_of_embeddings = embeddings.detach().cpu().float().numpy()
list_of_embeddings.shape

(15, 2560)

In [27]:
from sklearn.metrics.pairwise import cosine_similarity

print(cosine_similarity(list_of_embeddings[0].reshape(1,-1),list_of_embeddings[10].reshape(1,-1)))

[[0.6236255]]


In [33]:
trial_ = []
for i in range(1,15):
    cos_sim = cosine_similarity(list_of_embeddings[0].reshape(1,-1),list_of_embeddings[i].reshape(1,-1))
    trial_.append((i,cos_sim))

trial_
trial_.sort(key=lambda x : x[1],reverse=True)
trial_

[(1, array([[0.9225259]], dtype=float32)),
 (3, array([[0.8486296]], dtype=float32)),
 (2, array([[0.80480635]], dtype=float32)),
 (4, array([[0.75033975]], dtype=float32)),
 (5, array([[0.71809053]], dtype=float32)),
 (9, array([[0.71584827]], dtype=float32)),
 (8, array([[0.70898235]], dtype=float32)),
 (7, array([[0.6967458]], dtype=float32)),
 (12, array([[0.6667099]], dtype=float32)),
 (6, array([[0.65314204]], dtype=float32)),
 (10, array([[0.6236255]], dtype=float32)),
 (11, array([[0.6211858]], dtype=float32)),
 (13, array([[0.6118858]], dtype=float32)),
 (14, array([[0.6000807]], dtype=float32))]

In [38]:
print(f"Top 4 closest movie to {trial_list[0]} are: ")
for i,_ in trial_[:4]:
    print(trial_list[i],end=" | ")

Top 4 closest movie to Iron Man are: 
The Avengers | Avengers: Endgame | Spider-Man: No Way Home | Top Gun: Maverick | 